# Create helper methods to:
 1. Load the CoDocGen model to generate documentation for a function
 2. Method that uses the CoDocModel model to generate documentation of the given code (as string)
 3. Load the the-stack dataset and extract only the C++ and python code from it.
 4. load the github/tree-sitter to create abstract syntx trees of given code and lanugage
 5. Create ASTs for the given code.

# Install all the necessary packages

In [2]:
pip install transformers torch accelerate  datasets  tree-sitter tree-sitter-languages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.4/635.4 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 103.2 MB/s eta 0:00:00


In [ ]:

# Upgrade torchao to a compatible version
!pip install --upgrade torchao

In [3]:
from google.colab import userdata
import os

# Try to retrieve the Hugging Face API key from Colab's secrets manager
HUGGING_FACE_KEY = userdata.get('HUGGING_FACE_KEY')

if HUGGING_FACE_KEY is None:
    print("WARNING: Hugging Face API key (HUGGING_FACE_KEY) not found in Colab secrets.")

    # Prompt user for input if not found in secrets
    HUGGING_FACE_KEY = input("Please enter your Hugging Face API Key: ")
    if not HUGGING_FACE_KEY:
        print("Hugging Face API Key was not provided. Some models might not load correctly.")
        import sys
        sys.exit()
    else:
        os.environ['HF_TOKEN'] = HUGGING_FACE_KEY
        print("Hugging Face API key received from input.")
else:
    # Set the environment variable for Hugging Face if found in secrets
    os.environ['HF_TOKEN'] = HUGGING_FACE_KEY
    print("Hugging Face API key loaded successfully from secrets.")

Hugging Face API key loaded successfully from secrets.


# All models will be singleton so create base metaclass for the singleton

In [4]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):

        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(
                *args,
                **kwargs
            )

        return cls._instances[cls]

 # Singleton for the CodeGeneration

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch

class CodeDocumentationGenerator(metaclass=SingletonMeta):
    def __init__(self):
        """
        Constructor.

        IMPORTANT:
        Since SingletonMeta returns the same object every time,
        __init__ may be called multiple times.

        Therefore we guard against reinitialization.
        """

        if hasattr(self, "_initialized"):
            return

        self._initialized = True
        #self._model_name = "CoDoCGen/CoDoCGen-7B"
        self._model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"
        self._load_codocgen_model()


    def _load_codocgen_model(self):
      """Loads the CoDocGen model and tokenizer."""
      self._model = AutoModelForCausalLM.from_pretrained(self._model_name,
                                                  torch_dtype="auto",
                                                  device_map="auto")

      self._tokenizer = AutoTokenizer.from_pretrained(self._model_name)


    def generate_documentation(self, code: str, max_length=512, num_return_sequences=1, prompt=None) -> list:
      """
        Generates documentation for a given code snippet using the CoDoCGen model.

      Args:
          code (str): The code for which to generate documentation.
          max_length (int): The maximum length of the generated documentation.
          num_return_sequences (int): The number of different documentation sequences to generate.

      Returns:
          list: A list of generated documentation strings.
      """
      if prompt is None:
        prompt = f"generate good detailed documentation for what this software code does, do not include a pseudo code or example usage, just the intent of what the program should do, if it follows a design pattern, what actions to take under what conditions. The first line of the documentation should start with 'A software program in ProgLang, where ProgLan is the programming language of the program: {code}"
      else:
        prompt = f"{prompt}: {code}"

      messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
      ]
      text = self._tokenizer.apply_chat_template( messages, tokenize=False, add_generation_prompt=True)
      model_inputs = self._tokenizer([text], return_tensors="pt").to(self._model.device)

      generated_ids = self._model.generate(**model_inputs, max_new_tokens=512)
      generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids) ]

      response = self._tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
      return response


In [10]:
# Test the documentation generation
sample_code = """
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
"""

print("Generating documentation for the following code:")
print(sample_code)


Generating documentation for the following code:

def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)



In [11]:
generator = CodeDocumentationGenerator()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [12]:
print("\nGenerated Documentation 2:")
documentation = generator.generate_documentation(sample_code)
print(documentation)


Generated Documentation 2:
A software program written in Python, where the function is named `factorial`, calculates the factorial of a given non-negative integer `n`.

The function uses a recursive approach to compute the factorial. Here's a detailed explanation of its behavior:

### Intent of the Program

The `factorial` function is designed to compute the factorial of a non-negative integer `n`. The factorial of a number `n` (denoted as `n!`) is the product of all positive integers less than or equal to `n`. For example, the factorial of 5 is calculated as `5! = 5 × 4 × 3 × 2 × 1 = 120`.

### Design Pattern

This function follows a recursive design pattern, which is a common technique used in computer science for solving problems that can be broken down into smaller, similar subproblems.

### Actions Taken Under Different Conditions

1. **Base Case**:
   - If `n` is equal to 0, the function immediately returns 1. This is because the factorial of 0 is defined as 1. The base case ens

## Load and Filter Code Dataset

A function to load a `bigcode/the-stack` dataset and filter it to include only C++ and Python code. The `language` column has the type of laguage

In [13]:
from datasets import load_dataset, interleave_datasets

def load_and_filter_code_dataset(languages:list =None):
    """
    Loads a code dataset and filters it by specified languages.

    Args:
        dataset_name (str): The name of the dataset to load (e.g., "codeparrot/github-code").
        languages (list): A list of programming languages to filter by (e.g., ['C++', 'Python']).
                          If None, no language filtering is applied.

    Returns:
        datasets.Dataset: The filtered dataset.
    """
    dataset_name ="bigcode/the-stack-dedup"
    languages_and_dataset = [
                              {
                                  'language':'python',
                                  'dataset':None
                              },
                              {
                                  'language':'cpp',
                                  'dataset':None
                              }
                            ]

    print(f"Loading dataset: {dataset_name}")
    train_dataset = None

    for i in range(len(languages_and_dataset)):
      language = languages_and_dataset[i]['language']
      print(f"Fetching dataset for '{language}' language")
      # 2. Merge them into one combined stream
      # probabilities=[0.5, 0.5] mixes them evenly (1 Python, 1 C++, 1 Python...)
      languages_and_dataset[i]['dataset'] = load_dataset(dataset_name,
                                       data_dir = f"data/{language}",
                                       split="train",
                                       streaming=True,
                                        token=True)


    return languages_and_dataset

### Test: Loading and Filtering Code

Use the `load_and_filter_code_dataset` function to get only C++ and Python code from the `codeparrot/github-code` dataset.

In [14]:
try:
    cpp_python_dataset = load_and_filter_code_dataset()

    print(next(iter(cpp_python_dataset[0]['dataset'])))
    print(next(iter(cpp_python_dataset[1]['dataset'])))
    # print("\nFirst example from filtered dataset (showing language and a snippet of code):")
    # first_example_filtered = next(iter(cpp_python_dataset))
    # print(f"---\nLanguage: {first_example_filtered.get('lang', 'N/A')}\nCode Snippet: {first_example_filtered.get('content', 'N/A')[:200]}...")

    # The original intent was to get 5 examples, but for streaming datasets,
    # directly indexing or taking len() can be problematic. Iterating explicitly is safer.
    # Let's just confirm the first example for now.

except RuntimeError as e:
    if "Dataset scripts are no longer supported" in str(e):
        print(f"\nError loading dataset: {e}")
        print("\nIt appears the `codeparrot/github-code` dataset cannot be loaded directly via script anymore.")
        print("To fix this, please modify the `load_and_filter_code_dataset` function in cell `CnarbxKBNomR`.")
        print("You might need to specify a `config_name` (e.g., 'all' or 'code_x_m') and potentially use `streaming=True` if the dataset is very large, like so:")
        print("    `dataset = load_dataset(dataset_name, 'all', split=\"train\", streaming=True)`")
        print("Alternatively, you might need to find a different version of the dataset or a more compatible dataset for demonstration purposes.")
    else:
        raise e

Loading dataset: bigcode/the-stack-dedup
Fetching dataset for 'python' language


README.md:   0%|          | 0.00/19.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/144 [00:00<?, ?it/s]

Fetching dataset for 'cpp' language


Resolving data files:   0%|          | 0/110 [00:00<?, ?it/s]

{'hexsha': 'd99a1e98eccb58cbc0c0cef6e9e6702f33461b0e', 'size': 5886, 'ext': 'py', 'lang': 'Python', 'max_stars_repo_path': 'public_data/serializers.py', 'max_stars_repo_name': 'MTES-MCT/sparte', 'max_stars_repo_head_hexsha': '3b8ae6d21da81ca761d64ae9dfe2c8f54487211c', 'max_stars_repo_licenses': ['MIT'], 'max_stars_count': None, 'max_stars_repo_stars_event_min_datetime': None, 'max_stars_repo_stars_event_max_datetime': None, 'max_issues_repo_path': 'public_data/serializers.py', 'max_issues_repo_name': 'MTES-MCT/sparte', 'max_issues_repo_head_hexsha': '3b8ae6d21da81ca761d64ae9dfe2c8f54487211c', 'max_issues_repo_licenses': ['MIT'], 'max_issues_count': 3, 'max_issues_repo_issues_event_min_datetime': '2022-02-10T11:47:58.000Z', 'max_issues_repo_issues_event_max_datetime': '2022-02-23T18:50:24.000Z', 'max_forks_repo_path': 'public_data/serializers.py', 'max_forks_repo_name': 'MTES-MCT/sparte', 'max_forks_repo_head_hexsha': '3b8ae6d21da81ca761d64ae9dfe2c8f54487211c', 'max_forks_repo_licenses'

In [15]:
print("\nFirst example from cpp_python_dataset (using next(iter())):\n")
first_example = next(iter(cpp_python_dataset))
print(first_example)


First example from cpp_python_dataset (using next(iter())):

{'language': 'python', 'dataset': IterableDataset({
    features: ['hexsha', 'size', 'ext', 'lang', 'max_stars_repo_path', 'max_stars_repo_name', 'max_stars_repo_head_hexsha', 'max_stars_repo_licenses', 'max_stars_count', 'max_stars_repo_stars_event_min_datetime', 'max_stars_repo_stars_event_max_datetime', 'max_issues_repo_path', 'max_issues_repo_name', 'max_issues_repo_head_hexsha', 'max_issues_repo_licenses', 'max_issues_count', 'max_issues_repo_issues_event_min_datetime', 'max_issues_repo_issues_event_max_datetime', 'max_forks_repo_path', 'max_forks_repo_name', 'max_forks_repo_head_hexsha', 'max_forks_repo_licenses', 'max_forks_count', 'max_forks_repo_forks_event_min_datetime', 'max_forks_repo_forks_event_max_datetime', 'content', 'avg_line_length', 'max_line_length', 'alphanum_fraction'],
    num_shards: 144
})}


### Generate Abstract Syntax Tree (AST) with Tree-sitter

To generate an Abstract Syntax Tree (AST) for code, we can use the `tree-sitter` library. This involves installing the library and specific language parsers.

In [16]:
!pip install tree-sitter-cpp tree-sitter-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 12.5 MB/s eta 0:00:00


In [17]:
from tree_sitter import Language, Parser
import tree_sitter_cpp as tscpp

cpp_language = Language(tscpp.language())

parser = Parser()
parser.language = cpp_language

code = b"""
class Singleton
{
public:
    static Singleton& getInstance();
};
"""

tree = parser.parse(code)

print(tree.root_node)

(translation_unit (class_specifier name: (type_identifier) body: (field_declaration_list (access_specifier) (field_declaration (storage_class_specifier) type: (type_identifier) declarator: (reference_declarator (function_declarator declarator: (field_identifier) parameters: (parameter_list)))))))


### Function generate the AST Given the code
The generate_ast_with_tree_sitter function will use the manually loaded Language objects and generate AST for the given code.
Expect input to be a string or bytes. If string then convert to tbytes before processing with tree sitter.

In [18]:
from tree_sitter import Tree
from typing import Union

def generate_ast_with_tree_sitter(code: Union[str, bytes], language_name: str):
    """
    Generates an Abstract Syntax Tree (AST) for a given code snippet
    using tree-sitter for the specified language.

    Args:
        code (Union[str, bytes]): The code for which to generate the AST. Can be a string or bytestring.
        language_name (str): The name of the programming language (e.g., 'python', 'cpp').

    Returns:
        tree_sitter.Tree: The generated AST.
    """

    from tree_sitter import Language, Parser
    import tree_sitter_cpp as tscpp
    import tree_sitter_python as tspy

    # Check if the language is supported
    if language_name.lower() not in ['cpp', 'c++', 'python']:
      raise ValueError(f"Supported programming languages are 'C++', 'Python'. '{language_name}' is not supported!")

    # Encode string to bytes if necessary
    if isinstance(code, str):
        code = code.encode('utf-8')

    # Load the language dynamically using tree-sitter-languages.
    # This function directly returns a tree_sitter.Language object.
    if language_name.lower() in ['cpp', 'c++']:
      prog_language = Language(tscpp.language())

    elif language_name.lower() == 'python':
      prog_language = Language(tspy.language())

    else:
      raise ValueError(f"Runtime environment error for language {language_name}")


    parser = Parser()
    parser.language = prog_language

    tree = parser.parse(code)
    return tree

def print_ast_tree(tree: Tree, indent=0):
    """
    Helper function to print AST nodes recursively with indentation.
    """
    def _print_node_recursive(node, current_indent):
        print(f"{_get_indent_string(current_indent)}{node.type} [start={node.start_point}, end={node.end_point}]")
        for child in node.children:
            _print_node_recursive(child, current_indent + 1)

    def _get_indent_string(current_indent):
        return '  ' * current_indent

    _print_node_recursive(tree.root_node, indent)

### Test: Generating ASTs

Let's demonstrate `generate_ast_with_tree_sitter` with a Python function and a C++ snippet.

In [19]:
# Example 1: Python Code
python_code = """
def greet(name):
    print(f"Hello, {name}!")
"""

print("Generating AST for Python code:")
python_ast = generate_ast_with_tree_sitter(python_code, 'python')
if python_ast:
    print_ast_tree(python_ast)

print("\n" + "="*50 + "\n")

# Example 2: C++ Code
cpp_code = b"""
#include <iostream>

int main() {
    std::cout << "Hello from C++!" << std::endl;
    return 0;
}
"""

print("Generating AST for C++ code:")
cpp_ast = generate_ast_with_tree_sitter(cpp_code, 'cpp')
if cpp_ast:
    print_ast_tree(cpp_ast)

Generating AST for Python code:
module [start=Point(row=1, column=0), end=Point(row=3, column=0)]
  function_definition [start=Point(row=1, column=0), end=Point(row=2, column=28)]
    def [start=Point(row=1, column=0), end=Point(row=1, column=3)]
    identifier [start=Point(row=1, column=4), end=Point(row=1, column=9)]
    parameters [start=Point(row=1, column=9), end=Point(row=1, column=15)]
      ( [start=Point(row=1, column=9), end=Point(row=1, column=10)]
      identifier [start=Point(row=1, column=10), end=Point(row=1, column=14)]
      ) [start=Point(row=1, column=14), end=Point(row=1, column=15)]
    : [start=Point(row=1, column=15), end=Point(row=1, column=16)]
    block [start=Point(row=2, column=4), end=Point(row=2, column=28)]
      expression_statement [start=Point(row=2, column=4), end=Point(row=2, column=28)]
        call [start=Point(row=2, column=4), end=Point(row=2, column=28)]
          identifier [start=Point(row=2, column=4), end=Point(row=2, column=9)]
          ar

# Create LORA of codegen-multi-350B for code generation where the input is code documentation and output is the code.

In [ ]:
# Load the Codegen-350M-multi model
# Lora Adapt
# For the C++ and python code, get one, get the documentation, give model the documentation and let it generate the code.
# The generated code must be same as the code for which the documetation was generated.

# The second set of train is where we give documentation of a python code and have the model generate C++ Code:
# 1. Take Python code.
# 2. generate documentation,
# 3. Generated documentation input to model to genreate C++ Code
# 4. C++ Code as input to model to generate Python Code
# 5. Python code comparison gives the loss.


In [ ]:
# Load codegen-350m-multi model from hugging face


In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name_codegen = "Salesforce/codegen-350M-multi"

# Load tokenizer and model for codegen
tokenizer_codegen = AutoTokenizer.from_pretrained(model_name_codegen)
model_codegen = AutoModelForCausalLM.from_pretrained(model_name_codegen)
for name, module in model_codegen.named_modules():
    print(name)

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



transformer
transformer.wte
transformer.drop
transformer.h
transformer.h.0
transformer.h.0.ln_1
transformer.h.0.attn
transformer.h.0.attn.attn_dropout
transformer.h.0.attn.resid_dropout
transformer.h.0.attn.qkv_proj
transformer.h.0.attn.out_proj
transformer.h.0.mlp
transformer.h.0.mlp.fc_in
transformer.h.0.mlp.fc_out
transformer.h.0.mlp.act
transformer.h.0.mlp.dropout
transformer.h.1
transformer.h.1.ln_1
transformer.h.1.attn
transformer.h.1.attn.attn_dropout
transformer.h.1.attn.resid_dropout
transformer.h.1.attn.qkv_proj
transformer.h.1.attn.out_proj
transformer.h.1.mlp
transformer.h.1.mlp.fc_in
transformer.h.1.mlp.fc_out
transformer.h.1.mlp.act
transformer.h.1.mlp.dropout
transformer.h.2
transformer.h.2.ln_1
transformer.h.2.attn
transformer.h.2.attn.attn_dropout
transformer.h.2.attn.resid_dropout
transformer.h.2.attn.qkv_proj
transformer.h.2.attn.out_proj
transformer.h.2.mlp
transformer.h.2.mlp.fc_in
transformer.h.2.mlp.fc_out
transformer.h.2.mlp.act
transformer.h.2.mlp.dropout
tran

### LORA Adaptation for `codegen-350M-multi`

Set up Low-Rank Adaptation (LORA) for the `codegen-350M-multi` model. This allows us to fine-tune the model efficiently without modifying all its parameters. We'll specify the LORA configuration, including the rank (`r`), alpha (`lora_alpha`), dropout (`lora_dropout`), and target modules (the layers to apply LORA to).


In [22]:
from peft import LoraConfig, get_peft_model, TaskType


# Define LORA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # For Causal Language Modeling
    inference_mode=False,
    r=8, # Rank of the update matrices, Should this be 16? TODO
    lora_alpha=32, # Scaling factor, should this be 32: TODO
    lora_dropout=0.1, # Dropout probability, should this be small 0.05: TODO
    target_modules=["qkv_proj", "out_proj"] # Apply LORA to query, key, value and out projections
)

# Apply LORA to the codegen model
model_codegen_lora = get_peft_model(model_codegen, lora_config)

print("LORA adapted model summary:")
model_codegen_lora.print_trainable_parameters()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


LORA adapted model summary:
trainable params: 983,040 || all params: 357,695,488 || trainable%: 0.2748


### Prepare Training Data for LORA

To train the LORA model to generate code from documentation, we need a dataset of (documentation, code) pairs. We will leverage the `CodeDocumentationGenerator` to create documentation for code snippets extracted from the `cpp_python_dataset`. We'll take a small sample due to the streaming nature of the dataset and for demonstration purposes.

The input to our LORA model will be `documentation_text + EOS_TOKEN + code_text`, where the model will be trained to generate `code_text` given `documentation_text`.

In [23]:
from datasets import Dataset

# Initialize the documentation generator (already done in a previous cell, but re-initialize for clarity if needed)
generator = CodeDocumentationGenerator()

def prepare_data_for_lora(dataset, num_samples=10):
    data = []
    print(f"Preparing {num_samples} samples for LORA training...")
    # Iterate through both Python and C++ datasets
    for lang_data in dataset:
        lang_name = lang_data['language']
        count = 0
        for example in lang_data['dataset']:
            if count >= num_samples / 2: # Take half samples from each language
                break
            code = example.get('content')
            if code and len(code) > 50 and len(code) < 1000: # Filter for reasonable code lengths
                try:
                    # Generate documentation for the code
                    documentation = generator.generate_documentation(code)
                    # Format for training: documentation -> code
                    formatted_text = f"Documentation: {documentation}\nCode: {code}{tokenizer_codegen.eos_token}"
                    data.append({"text": formatted_text})
                    count += 1
                    if count % 5 == 0: print(f"  Processed {count} {lang_name} samples.")
                except Exception as e:
                    # print(f"Skipping sample due to error in documentation generation: {e}")
                    continue
    return Dataset.from_list(data)

# Assuming cpp_python_dataset is already loaded from previous steps
# If not, you might need to run the `load_and_filter_code_dataset` cell again.
training_dataset = prepare_data_for_lora(cpp_python_dataset, num_samples=20)

print("\nExample of prepared training data:")
print(training_dataset[0]['text'])

Preparing 20 samples for LORA training...
  Processed 5 python samples.
  Processed 10 python samples.


KeyboardInterrupt: 

In [ ]:
for i in range(5):
  print("\n*********\n",training_dataset[i]['text'])

### Train the LORA Model

Set up the `Trainer` from the `transformers` library to fine-tune our LORA-adapted `codegen` model.
Define `TrainingArguments` to control the training process, such as the number of epochs, learning rate, and logging strategy.
Use a `DataCollatorForLanguageModeling` to handle batching and masking for language modeling tasks.

To save the best model based on validation accuracy , you would typically include a validation set and a custom `TrainerCallback` or configure `save_strategy='epoch'` and `load_best_model_at_end=True` with a specified `metric_for_best_model` in `TrainingArguments`.


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Tokenize the training dataset
def tokenize_function(examples):
    return tokenizer_codegen(examples["text"], truncation=True, max_length=512)

tokenized_training_dataset = training_dataset.map(tokenize_function, batched=True)

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer_codegen, mlm=False)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./codegen_lora_results",
    per_device_train_batch_size=2, # Adjust based on GPU memory
    gradient_accumulation_steps=4, # Increase if batch size is small
    num_train_epochs=3, # Number of training epochs
    learning_rate=2e-4,
    logging_dir="./codegen_lora_logs",
    logging_steps=10,
    save_strategy="epoch", # Save checkpoint every epoch
    save_total_limit=1, # Only keep the best model
    # evaluation_strategy="epoch", # Uncomment if you have a validation set
    # load_best_model_at_end=True, # Uncomment if you have a validation set
    # metric_for_best_model="eval_loss", # Uncomment if you have a validation set
)

# Initialize Trainer
trainer = Trainer(
    model=model_codegen_lora,
    args=training_args,
    train_dataset=tokenized_training_dataset,
    # eval_dataset=tokenized_validation_dataset, # Uncomment if you have a validation set
    data_collator=data_collator,
)

# Start training
print("Starting LORA training...")
trainer.train()
print("LORA training complete.")

# Save the final LORA model (or the best model if validation is used)
model_codegen_lora.save_pretrained("codegen_lora_adapter")
print("LORA adapter model saved to 'codegen_lora_adapter'.")